# Streaming Data with Apache Spark

Streaming data introduces a new level of complication.
Examples include:
- Social Media Feeds
- Sensor readings
- Log data

Handling this data uses either one of these approaches:
- Recompute
- Incremental Processing

## Spark Structured Streaming
The key is to handle live data streams as unbounded, continously growing tables where each incoming data item is appended as a new row.
<br>
To be considered valid for streaming:
- Append-only requirement-- existing data cannot be modified
<br>
Integrates with Delta Lake using DataStreamWriter and DataStreamReader in PySpark <br>
`streamDF = spark.readStream.table("src_table")`<br>
`streamDF.writeStream.table("target_table")`<br>

### Example 
Two Delta Lake tables 
- table_1
- table_2

**Goal:** Continuously stream data from table_1 to table_2 appending new records into table_2 every two minutes.<br>
```python
streamDF = spark.readStream.table("table_1")
streamDF.writeStream 
          .trigger(processingTime="2 minutes")
          .outputMode("append")
          .option("checkpointLocation", "/path")
          .table("table_2")
```

Checkpoints are used to store metadata about the streaming job including current state and progress
They can also be used to recover the streaming job from failures and maintain its state across restarts.

## Streaming Query Configurations
### Trigger Intervals
- Continuous-- fixed-intervals (default 500ms)
- Triggered-- processes all available data in multiple micro-batches then stops automatically
  - Once-- all at once in a single micro-batch (OOM memory issues, large volume of data issues)[DEPRECIATED]
  - availableNow-- also facilitates batch processing of all currently available data, but address the single large micro-batch issue by splitting it up

### Output Modes
- Append-- only newly received rows are appended
- Complete-- target table is overwritten with each batch

### Structured Streaming Guarantees
- Fault Recovery-- node crashes/network issues can resume processing with checkpointing and write-ahead logs
- Exactly-once Semantics-- each record will be processed exactly once (dedupe from data source)

## Unsupported operations
- Sorting
- Deduplication

# Implementing Structured Streaming
Let's now practically implement Spark Structured Streaming for incremental data processing.
<p>First, run our School-Setup notebook to configure our environment.</p>

In [0]:
%run ./School-Setup

Structured streaming affords high-level APIs for both SQL and Python. Regardless, you must always start with <br>
```python
stream_df = spark.readStream.table("courses")
```

In [0]:
stream_df = spark.readStream.option("checkpointLocation", "/tmp/checkpoint").table("courses")

## Streaming Data Manipulations in SQL
Convert streaming DF into format that SQL can interpret and query. So, we'll register it as a temp view.

In [0]:
stream_df.createOrReplaceTempView("courses_streaming_tmp_vw")

Creating a temporary view from a streaming dataframe creates a streaming temporary view.<br>
Thus, facilitating SQL transformations. 

In [0]:
display(spark.sql("SELECT * FROM courses_streaming_tmp_vw"), checkpointLocation = "/Workspace/users/bpatricka@live.com/tmp/checkpoints")

In practice, you won't typically display the results of streaming queries in notebooks unless there is a need to inspect them during development.

### Applying Transformations

In [0]:
display(spark.sql("SELECT instructor, count(course_id) AS total_courses  FROM courses_streaming_tmp_vw GROUP BY instructor"), checkpointLocation = "/Workspace/users/bpatricka@live.com/tmp/checkpoints")

### Persisting stream data
Persisting data to a durable storage involves returning our logic back to the PySpark df API. <br>
First, let's create a temp view to encapsulate our desired output.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW instructor_counts_tmp_vw AS (
  SELECT instructor, COUNT(course_id) AS total_courses
  FROM courses_streaming_tmp_vw
  GROUP BY instructor
)

In [0]:
# now lets access the output data using PySpark DF
result_stream_df = spark.table("instructor_counts_tmp_vw")

Spark differentiates between streaming and static dataframes (i.e. spark.readStream as opposed to spark.read)

In [0]:
result_stream_df.writeStream \
                .trigger(processingTime="3 seconds") \
                .outputMode("complete") \
                .option("checkpointLocation", "/Workspace/users/bpatricka@live.com/tmp/checkpoint/instructor_counts") \
                .table("instructor_counts")

In [0]:
%sql
SELECT * FROM instructor_counts

In [0]:
%sql
-- Let's simulate some new data
INSERT INTO courses
VALUES ("C16", "Generative AI", "Pierre B.", "Computer Science", 25),
       ("C17", "Embedded Systems", "Julia S.", "Computer Science", 30),
       ("C18", "Virtual Reality", "Bernard M.", "Computer Science", 35)

REMEMBER: Shut down active streams as they will not terminate and will keep the cluster active as well.

#### Next scenario
We'll introduce a set of courses taught by new instructors and incorporate them into our source table

In [0]:
%sql
INSERT INTO courses
VALUES ("C19", "Compiler Design", "Sophie B.", "Computer Science", 25),
       ("C20", "Signal Processing", "Sam M.", "Computer Science", 30),
       ("C21", "Operating Systems", "Mark H.", "Computer Science", 35)

In [0]:
result_stream_df.writeStream \
                .trigger(availableNow=True) \
                .outputMode("complete") \
                .option("checkpointLocation", "/Workspace/users/bpatricka@live.com/tmp/checkpoint/instructor_counts") \
                .table("instructor_counts") \
                .awaitTermination()

In [0]:
%sql
SELECT * FROM instructor_counts

## Streaming Data Manipulations in Python
No need for temp object or view, apply data processing directly on the streaming DataFrame

In [0]:
import pyspark.sql.functions as F

# streaming dataframes are immutable like static dfs, which is why we create a new df anytime a transformation is made
output_stream_df = (stream_df.groupBy("instructor")).agg(F.count("course_id").alias("total_courses"))
display(output_stream_df, checkpointLocation = "/Workspace/users/bpatricka@live.com/tmp/checkpoints/py_total_courses")

Now to examine how we will persist data from this stream.

In [0]:
(output_stream_df.writeStream     
                 .trigger(availableNow=True) \
                 .outputMode("complete") \
                 .option("checkpointLocation", \
                   "/Workspace/users/bpatricka@live.com/databricks_repo/checkpoints/instructor_counts_py") \
                 .table("instructor_counts_py") \
                 .awaitTermination())

In [0]:
%sql
SELECT * FROM instructor_counts_py

In [0]:
# or we can use pyspark to query now, too
display(spark.read.table("instructor_counts_py"))

# Incremental Data Ingestion
## Introducing Data Ingestion
Incremental data ingestion involves loading only files newly received since the last data ingestion cycle.
- COPY INTO
- Auto Loader

## COPY INTO Command
Idempotent and incremental manner (dedupe new files)
```sql
1 COPY INTO my_table
2 FROM '/path/to/files'
3 FILEFORMAT = <format>
4 FORMAT_OPTIONS (<format options>)
5 COPY_OPTIONS (<copy options>)
```
<br><br>
```sql
COPY INTO my_table
FROM '/path/to/files'
FILEFORMAT = CSV
FORMAT_OPTIONS ('delimiter' = '|',
             'header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
```

## Auto Loader
Leverages Structured Streaming in Spark to offer scalability and handling of billions of files and real-time ingestions of millions of files per hour. It also employs checkpointing.

readStream and writeStream methods are utilized:<br>
```python
spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", <source_format>)
         .load(’/path/to/files’)
     .writeStream
         .option("checkpointLocation", <checkpoint_directory>)
         .table(<table_name>)
```

schema can be inferred or explicitly defined, datatypes are inferred from parquet, but string for others CSV, JSON

```python
spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", <source_format>)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", <schema_directory>)
        .load(’/path/to/files’)
    .writeStream
        .option("checkpointLocation", <checkpoint_directory>)
        .option("mergeSchema", "true")
        .table(<table_name>)
```

## Comparison of Ingestion Mechanisms
- COPY INTO 
  - Volume of incoming files is relatively small (1000s)
- Auto Loader
  - Volume is order of millions
  - Efficiently splits processing into batches
  - Ideal for high data velocity and volume as well as cloud object storage

## Auto Loader in Action
Let's get practical! We'll use Auto Loader for incremental data ingestion from files.

In [0]:
%run ./School-Setup

Auto Loader:
JSON files -> Target Delta Table

In [0]:
# before setting up auto loader, let's inspect our source directory
files = dbutils.fs.ls(f"{dataset_school}/enrollments-json-raw")
display(files)

In [0]:
# now we'll setup our Auto Loader to deal with any new or existing files in this directory
(spark.readStream \
        .format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .option("cloudFiles.inferColumnTypes", "true") \
        .option("cloudFiles.schemaLocation", "/Workspace/users/bpatricka@live.com/mnt/DEA-Book/checkpoints/enrollments") \
        .load(f"{dataset_school}/enrollments-json-raw") \
).writeStream \
        .option("checkpointLocation", "/Workspace/users/bpatricka@live.com/mnt/DEA-Book/checkpoints/enrollments") \
        .table("enrollments_updates")

In [0]:
%sql
SELECT * FROM enrollments_updates

### Observing Auto Loader
We can simulate an external system adding new files to our directory, we have a helper function from School-Setup

In [0]:
load_new_data()

In [0]:
# we've added two more files let's take a look
files = dbutils.fs.ls(f"{dataset_school}/enrollments-json-raw")
display(files)

In [0]:
%sql
SELECT COUNT(*) FROM enrollments_updates

### Exploring table history

In [0]:
%sql
DESCRIBE HISTORY enrollments_updates

### Clean up!
- Drop tables
- Remove Chechpoint
- ...and oh ya that streaming job is still running!

In [0]:
%sql
-- now that our streaming job stopped, lets drop the tables
DROP TABLE IF EXISTS enrollments_updates

In [0]:
# then we'll remove the checkpoints
dbutils.fs.rm("/Workspace/users/bpatricka@live.com/mnt/DEA-Book/checkpoints/enrollments", True)

# Medallion Architecture
## Introducing Medallion Architecture
Gradually enhance data as it move through the structure.
### The Layered Approach
Bronze => Silver => Gold

- Bronze
  - Raw data-- all types of data files like structured or operational databases also destinations for real-time data ingestion
  - Captured and stored, regardless of source or quality
- Silver
  - Data cleansing, normalization, and validation
  - incorrect or irrelevant data are filtered out
  - inconsistencies resolved
  - enrichment via joining
  - designed to prepare data for analytical tasks that require higher degree of integrity and accuracy
- Gold
  - Facilitates high-level business analytics and intelligence
  - Aggregated and Summarized to support business needs and decisions
  - Transformations ensure data readiness for reporting, dashboarding, and advanced analytics in ML/AI
### Benefits of Medallion
- Simplicity
- Incremental ETL
- Hybrid Workloads
- Table reconstruction

## Building Medallion Architectures
Let's start now by looking at a practical example use students, enrollments, and courses to build and implement this Medallion Architecture.

In [0]:
%run ./School-Setup


In [0]:
files = dbutils.fs.ls(f"{dataset_school}/enrollments-json-raw")
display(files)

We have three files awaiting ingestion to the Bronze layer.

First, we need to configure our Auto Loader stream to process the input files and load the data into a Delta Lake table:

In [0]:
import pyspark.sql.functions as F

(
    spark.readStream \
        .format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .option("cloudFiles.inferColumnTypes", "true") \
        .option("cloudFiles.schemaLocation", \
            f"{checkpoint_path}/enrollments_bronze") \
        .load(f"{dataset_school}/enrollments-json-raw") \
        .select("*",
                F.current_timestamp().alias("arrival_time"),
                F.input_file_name().alias("source_file"))
).writeStream \
        .format("delta") \
        .option("checkpointLocation", \
                f"{checkpoint_path}/enrollments_bronze") \
        .outputMode("append") \
        .table("enrollments_bronze")

In [0]:
%sql
SELECT * FROM enrollments_bronze

In [0]:
%sql
SELECT COUNT(1) FROM enrollments_bronze

Let's simulate some new data

In [0]:
load_new_data()

Let's create a static lookup table

In [0]:
students_lookup_df = spark.read \
                            .format("json") \
                            .load(f"{dataset_school}/students-json")
display(students_lookup_df)

- [x] Bronze Layer [check!]

Now lets refine and enhance data acquired from the bronze by:
- adding contextual information
- formatting values
- performing data quality checks

**Goal:** Ensure data is clean, structured, and optimized for downstream processing and analysis

In [0]:
# this starts with initiating a streaming read operation on the enrollments_bronze table
# then apply a series of transformations to enrich and refine the data
enrollments_enriched_df = (spark.readStream
                           .table("enrollments_bronze")
                           .where("quantity > 0")
                           .withColumn("formatted_timestamp",
                                       F.from_unixtime("enroll_timestamp",
                                                       "yyyy-MM-dd HH:mm:ss").cast("timestamp"))
                           .join(students_lookup_df, "student_id")
                           .select("enroll_id", "quantity", "student_id", "email", "formatted_timestamp", "courses"))

In [0]:
# lets finalize this data workflow by persisting the data to a silver table
enrollments_enriched_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{checkpoint_path}/enrollments_silver") \
    .outputMode("append") \
    .table("enrollments_silver")

And so, we've just set up a continuous streaming write into enrollments_silver
Which establishes our pipeline from bronze -> silver <br>
enrollments_enriched_df is created from readstream
and then set to writeStream as append (new records)

In [0]:
%sql
SELECT * FROM enrollments_silver

In [0]:
# lets demonstrate the full power of our bronze to silver pipeline by simuating new data
load_new_data()

Thats pretty neat, we can observe the new data flowing from our bronze into our silver.

In [0]:
%sql
SELECT COUNT(*) FROM enrollments_silver

Now, we're ready to go Gold.<br>
'*Don't believe me, just watch.*

### Advancing Gold
We provide high-level aggregations and summaries

**Goal:** Create an aggregate table that summarizes daily number of course enrollments per student
- [ ] Initiate streaming read operation on enrollments_silver
- [ ] Perform necessary transformations to aggregate the data by student_id, email and day

In [0]:
enrollments_agg_df = spark.readStream \
    .table("enrollments_silver") \
    .withColumn("day", F.date_trunc("DD", "formatted_timestamp")) \
    .groupBy("student_id", "email", "day") \
    .agg(F.sum("quantity").alias("courses_count")) \
    .select("student_id", "email", "day", "courses_count")

In [0]:
# now we have our aggregated dataframe we can use writeStream
enrollments_agg_df.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", f"{checkpoint_path}/daily_student_courses") \
    .trigger(availableNow=True) \
    .table("daily_student_courses")

In [0]:
%sql
SELECT * FROM daily_student_courses

In [0]:
# lets again simulate more data to see our full pipeline
load_new_data()

It's important to remember that you have to rerun the streaming query to update the gold layer table

In [0]:
%sql
-- after executing the streaming write query for daily_student_courses we can query it
SELECT * FROM daily_student_courses

Okay, the fun and games are over kid.<br>
Let's pack 'er up and call it a day. <br>
Don't forget to put out the fire before you leave. (close and terminate active streams in our session)

In [0]:
for s in spark.streams.active:
    print("Stopping stream: ", s.name)
    s.stop()
    s.awaitTermination()